# RiskSequencer — 01 EDA

Exploratory analysis for the behavioral fraud model. Works on **either** the
synthetic generator or the real IEEE-CIS data (drop `transactions.parquet` in
`data/raw/` via `python -m data.ieee_cis`).

Goals (build plan Phase 1.2):
- Profile class imbalance
- Transaction velocity, time gaps, amount distributions
- Top behavioral features by mutual information with the label
- Decide imbalance strategy (we use `pos_weight`)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # make project root importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import RAW_DIR, FEATURE_COLUMNS, LABEL_COL, MONITORED_FEATURES
pd.set_option("display.max_columns", 50)

## 1. Load data

Uses the real IEEE-CIS parquet if present, else falls back to the synthetic generator.

In [ ]:
parquet = RAW_DIR / "transactions.parquet"
if parquet.exists():
    raw = pd.read_parquet(parquet)
    source = "IEEE-CIS (real)"
else:
    from data.synthetic import generate_transactions
    raw = generate_transactions(n_users=3000, seed=42)
    source = "synthetic"

print(f"Source: {source}")
print(f"Rows: {len(raw):,}  |  Users: {raw['user_id'].nunique():,}")
raw.head()

## 2. Class imbalance

Fraud is typically 1–3% of transactions. This dictates the loss weighting.

In [ ]:
row_rate = raw[LABEL_COL].mean()
user_rate = raw.groupby("user_id")[LABEL_COL].max().mean()
print(f"Row-level fraud rate : {row_rate:.3%}")
print(f"User-level fraud rate: {user_rate:.3%}  (label used by the sequence model)")

pos = raw[LABEL_COL].sum(); neg = len(raw) - pos
print(f"pos_weight (neg/pos) ≈ {neg/max(pos,1):.1f}")

raw[LABEL_COL].value_counts().plot(kind="bar", title="Class balance (0=legit, 1=fraud)")
plt.show()

## 3. Transactions per user & time gaps

Sequence length is fixed at 50; how many users actually fill the window?

In [ ]:
counts = raw.groupby("user_id").size()
print(counts.describe())
plt.hist(counts.clip(upper=120), bins=40)
plt.axvline(50, color="r", ls="--", label="seq_len=50")
plt.title("Transactions per user"); plt.xlabel("# txns"); plt.legend(); plt.show()
print(f"Users with >=50 txns (full window): {(counts >= 50).mean():.1%}")

In [ ]:
# Inter-arrival gaps (hours) — fraud bursts should show much smaller gaps
g = raw.sort_values(["user_id", "timestamp"]).copy()
g["gap_h"] = g.groupby("user_id")["timestamp"].diff().dt.total_seconds() / 3600.0
for label, sub in g.groupby(LABEL_COL):
    print(f"label={label}: median gap = {sub['gap_h'].median():.2f}h")
g.boxplot(column="gap_h", by=LABEL_COL, showfliers=False)
plt.suptitle(""); plt.title("Inter-arrival gap by label"); plt.ylabel("hours"); plt.show()

## 4. Amount distribution

Account-takeover fraud often involves larger-than-usual amounts.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for label, sub in raw.groupby(LABEL_COL):
    ax[0].hist(np.log1p(sub["amount"]), bins=50, alpha=0.5, label=f"label={label}", density=True)
ax[0].set_title("log(1+amount) by label"); ax[0].legend()
raw.boxplot(column="amount", by=LABEL_COL, showfliers=False, ax=ax[1])
ax[1].set_title("amount by label"); plt.suptitle(""); plt.show()
print(raw.groupby(LABEL_COL)["amount"].describe()[["mean", "50%", "max"]])

## 5. Engineered features — mutual information with the label

Ranks the engineered features by how much they tell us about fraud. The three
`MONITORED_FEATURES` should rank highly — that's why they're the drift signals.

In [ ]:
from features.feature_pipeline import build_features
from sklearn.feature_selection import mutual_info_classif

feats = build_features(raw)
X = feats[FEATURE_COLUMNS].to_numpy()
y = feats[LABEL_COL].to_numpy()

mi = mutual_info_classif(X, y, random_state=0)
mi_s = pd.Series(mi, index=FEATURE_COLUMNS).sort_values(ascending=False)
print("Monitored features:", MONITORED_FEATURES)
mi_s.plot(kind="barh", figsize=(7, 6), title="Mutual information with fraud label")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
mi_s

## 6. Feature correlation

Highly correlated features add little; useful for pruning later.

In [ ]:
corr = feats[FEATURE_COLUMNS].corr()
plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(); plt.xticks(range(len(FEATURE_COLUMNS)), FEATURE_COLUMNS, rotation=90)
plt.yticks(range(len(FEATURE_COLUMNS)), FEATURE_COLUMNS); plt.title("Feature correlation")
plt.tight_layout(); plt.show()

## 7. Findings & decisions

> Fill in after running on the **real** IEEE-CIS data.

- **Imbalance:** row-level fraud ≈ ___%. Strategy: `pos_weight` in `BCEWithLogitsLoss`
  (no blind oversampling — per the hard constraints).
- **Velocity signal:** fraud inter-arrival gaps are (smaller / similar) → velocity
  features are (informative / not).
- **Amount signal:** fraud amounts skew (higher / similar).
- **Top MI features:** ___, ___, ___. Monitored features rank at #__, #__, #__.
- **Window coverage:** __% of users have ≥ 50 txns; the rest are left-padded.
- **Next:** run `python -m training.hyperparameter_search` then a full
  `training/train.py` run and record the held-out test AUC.